## 01 Exploratory Data Analysis & Cleaning

**Dataset:** IBM Telco Customer Churn - 7,043 customers


### Setup & Dataset overview

In [1]:
import sys
import warnings
import sqlite3
from pathlib import Path
import pandas as pd
warnings.filterwarnings('ignore')

# Requires running first: python download_data.py
ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / "data" / "telco_churn.csv")
print(df.shape)
print(df.dtypes)
df.head()

(7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Churn rate global

In [2]:
print('Churn distribution:')
print(df['Churn'].value_counts())
print()
print(f"Churn rate: {df['Churn'].value_counts(normalize=True)['Yes']*100:.2f}%")

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.54%


### SQL Analysis - Churn by contract type


In [ ]:
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)
conn = sqlite3.connect(":memory:")
df.to_sql("telco", conn, index=False, if_exists="replace")

q1 = '''
SELECT
    Contract,
    COUNT(*) AS n_customers,
    SUM(Churn_bin) AS n_churned,
    ROUND(AVG(Churn_bin)*100, 1) AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
    ROUND(SUM(MonthlyCharges)*12, 0) AS annual_revenue_at_risk
FROM telco
GROUP BY Contract
ORDER BY churn_rate_pct DESC'''
pd.read_sql(q1, conn)

,Contract,n_customers,n_churned,churn_rate_pct,avg_monthly_charges,annual_revenue_at_risk
0,Month-to-month,3875,1655,42.7,66.40,3087530.0
1,One year,1473,166,11.3,65.05,1149799.0
2,Two year,1695,48,2.8,60.77,1236070.0


Key finding: month-to-month customers churn at 14.5x the rate of two-year contracts.
Physical interpretation: no contractual friction -> low system inertia (small γ).

### SQL Analysis - Churn by tenure segment

In [4]:
q2 = '''
SELECT
    CASE WHEN tenure <= 6  THEN '0–6m'
         WHEN tenure <= 12 THEN '7–12m'
         WHEN tenure <= 24 THEN '13–24m'
         WHEN tenure <= 48 THEN '25–48m'
         ELSE '48m+' END AS tenure_segment,
    COUNT(*) AS total,
    ROUND(AVG(Churn_bin)*100, 1) AS churn_rate_pct
FROM telco
GROUP BY tenure_segment
ORDER BY MIN(tenure)'''
pd.read_sql(q2, conn)

,tenure_segment,total,churn_rate_pct
0,0–6m,1481,52.9
1,7–12m,705,35.9
2,13–24m,1024,28.7
3,25–48m,1594,20.4
4,48m+,2239,9.5


Key finding: 52.9% of customers churn in their first 6 months.
Physical interpretation: new customers have **low** initial energy E0 (E0 is dominated by the tenure term, which starts near 0 and rises with tenure), not high -- they haven't accumulated engagement yet, so they sit closest to the critical threshold and are most vulnerable to any perturbation.
The model must detect risk within the first 60–90 days to enable intervention.

### Numeric variable comparison : churners vs retained

In [5]:
q3 = '''
SELECT
    Churn,
    ROUND(AVG(tenure), 1) AS avg_tenure_months,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
    ROUND(AVG(TotalCharges), 2) AS avg_total_charges
FROM telco
GROUP BY Churn'''
pd.read_sql(q3, conn)

,Churn,avg_tenure_months,avg_monthly_charges,avg_total_charges
0,No,37.6,61.27,2549.91
1,Yes,18.0,74.44,1531.80


Insights:
1. Churners pay MORE per month ($74.44 vs $61.27) but stay LESS time.
2. The combination: high monthly charge + low tenure = high free energy.
3. avg_total_charges is LOWER for churners because they leave early.
4. Physical variable: TotalCharges ∝ ∫E(t)dt, integral of satisfaction over time

### Correlation matrix - numeric variables

In [6]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_bin']
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
corr = df[numeric_cols].corr().round(3)

print('Correlation matrix (numeric variables):')
print(corr)
print()
print('Key correlations with Churn_bin:')
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    print(f' {col}: {corr.loc[col, "Churn_bin"]:+.3f}')

Correlation matrix (numeric variables):
                tenure  MonthlyCharges  TotalCharges  Churn_bin
tenure           1.000           0.248         0.826     -0.352
MonthlyCharges   0.248           1.000         0.651      0.193
TotalCharges     0.826           0.651         1.000     -0.198
Churn_bin       -0.352           0.193        -0.198      1.000

Key correlations with Churn_bin:
 tenure: -0.352
 MonthlyCharges: +0.193
 TotalCharges: -0.198


tenure: -0.352 -> longer tenure = lower churn (strongest predictor)  
MonthlyCharges: +0.193 -> higher monthly charge = slightly higher churn   
TotalCharges: -0.198 -> higher total = lower churn (driven by tenure) 

In [7]:
# Export cleaned data
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)
df.to_csv(ROOT / 'data' / 'telco_clean.csv', index=False)
print(f'Saved telco_clean.csv - {len(df):,} rows')

Saved telco_clean.csv - 7,043 rows


In [8]:
sys.path.insert(0, str(ROOT))
from src.features import build_physical_features

df_features = build_physical_features(df)
df_features.to_csv(ROOT / 'data' / 'telco_features.csv', index=False)
print(f'Saved telco_features.csv - {len(df_features):,} rows, {len(df_features.columns)} columns')

Saved telco_features.csv - 7,043 rows, 41 columns


In [9]:
df_final = df_model = pd.get_dummies(df_features, columns=['Contract','PaymentMethod','InternetService'], drop_first=True)
df_final.to_csv(ROOT / 'data' / 'telco_final.csv', index=False)
print(f'Saved telco_final.csv - {len(df_final):,} rows, {len(df_final.columns)} columns')

Saved telco_final.csv - 7,043 rows, 45 columns


### Findings

| Observation | Physical interpretation |
|---|---|
| 52.9% churn in first 6 months | New customers: **low initial energy E0** (tenure term dominates E0 and hasn't accumulated yet) |
| Month-to-month: 42.2% vs 2.9% two-year | Low contractual friction -> **low γ** (damping coefficient) |
| Churners pay more ($74/mo) but stay less | High energy charge drives **faster relaxation to equilibrium** |
| Strong tenure–churn correlation (-0.35) | Long-tenured customers have settled into their equilibrium state |
| 11 customers with TotalCharges = NaN | tenure = 0 → excluded from physical model (undefined E0) |

-> These observations motivate the physical variable mapping in 02_physical_variables.ipynb